# Agentic AI Bootcamp
## Day 1 — Building Your First Reasoning Agent

---

**Course:** 5-Day Agentic AI Bootcamp  
**Day:** 1 of 5  
**Duration:** 2 hours  
**Platform:** University Assignment Assistant  

---

### What you will build today
By the end of this session you will have a **working AI agent** that:
- Receives any assignment question
- Plans a step-by-step approach using **Chain-of-Thought**
- Answers the question using **Claude**
- Critiques its own answer
- Remembers what it has already answered

### The 5-day arc
| Day | What we add |
|-----|-------------|
| **Day 1** ← you are here | Single reasoning agent with CoT |
| Day 2 | Tool use — web search, calculator, file reader |
| Day 3 | Multi-agent system with LangGraph |
| Day 4 | Persistent memory with ChromaDB + MCP tools |
| Day 5 | FastAPI + WhatsApp interface + Docker + monitoring |

> **Instructor tip:** Run each cell top-to-bottom. Every cell builds on the previous one.

## Today's Agenda

| Time | Topic |
|------|-------|
| 0:00 – 0:10 | Setup: install packages + configure API key |
| 0:10 – 0:25 | **Part 1:** What is an LLM? Your first API call |
| 0:25 – 0:45 | **Part 2:** System prompt engineering |
| 0:45 – 1:05 | **Part 3:** The PPARM Framework |
| 1:05 – 1:25 | **Part 4:** Chain-of-Thought vs Direct answering |
| 1:25 – 1:50 | **Part 5:** Lab — Build Assignment Agent v1 |
| 1:50 – 2:00 | Checkpoint: demo your agent to the class |

---
# Setup
Run the two cells below before anything else. This installs the Anthropic SDK and securely stores your API key for this session.

In [1]:
# Install dependencies
!pip install anthropic --quiet
print("Packages installed successfully.")

Packages installed successfully.


In [4]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Attempt to load .env from the project root, even in notebooks where __file__ is not defined.
def find_dotenv_path():
    # Try to detect .env in common locations relative to CWD or notebook hierarchy
    candidates = [
        Path.cwd() / ".env",
        Path.cwd().parent / ".env",
        Path.cwd().parent.parent / ".env"
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None

dotenv_path = find_dotenv_path()
if dotenv_path:
    load_dotenv(dotenv_path=dotenv_path)
    print(f"Loaded .env from: {dotenv_path}")
else:
    print(
        "Warning: No .env file found in standard locations such as current, parent, or grandparent directories."
    )

if "ANTHROPIC_API_KEY" not in os.environ or not os.environ["ANTHROPIC_API_KEY"]:
    raise OSError(
        "Could not find ANTHROPIC_API_KEY in your .env file at project root or common parent directories. "
        "Please add ANTHROPIC_API_KEY=YOUR_KEY to .env."
    )
else:
    print("API key loaded from .env.")

import anthropic
client = anthropic.Anthropic()
MODEL  = "claude-haiku-4-5"   # fast + cheap — perfect for learning
print(f"Client ready. Model: {MODEL}")

Loaded .env from: /home/administrator/Desktop/Bootcamp/.env
API key loaded from .env.
Client ready. Model: claude-haiku-4-5


> **How to add your key to Colab Secrets (recommended)**  
> 1. Click the **key icon** (🔑) in the left sidebar  
> 2. Click **+ Add new secret**  
> 3. Name: `ANTHROPIC_API_KEY` | Value: your key  
> 4. Toggle **Notebook access** ON  
> 5. Re-run the cell above
>
> Never paste your API key directly into a cell — it gets saved in the notebook file.

---
# Part 1 — What is an LLM? Your First API Call

## The Big Picture

An **LLM (Large Language Model)** is a neural network trained on billions of text examples. It predicts the most likely next token given everything before it.

```
  You send:  [ System prompt ] + [ User message ]
              ↓
         LLM processes tokens
              ↓
  You get:   [ Assistant reply ]
```

That's it. Everything we build this week is just clever ways of **structuring that input** and **chaining those calls** together.

### The three roles
| Role | Who says it | Purpose |
|------|-------------|---------|
| `system` | You (developer) | Defines the agent's identity, rules, and format |
| `user` | The end user | The actual question or task |
| `assistant` | The model | The response (also used to prime multi-turn chats) |

In [5]:
# Your first LLM call — bare minimum
response = client.messages.create(
    model=MODEL,
    max_tokens=256,
    messages=[
        {"role": "user", "content": "What is an AI agent? Explain in 3 sentences."}
    ],
)

print(response.content[0].text)

# AI Agent

An AI agent is a software system that perceives its environment through inputs (like data or sensor readings) and takes actions to achieve specific goals. It uses algorithms and sometimes machine learning to make decisions autonomously without requiring constant human direction. Examples range from simple chatbots that respond to queries to complex systems like autonomous vehicles that navigate and make real-time decisions.


### Anatomy of the response object
The API returns more than just text. Let's look inside it.

In [6]:
response = client.messages.create(
    model=MODEL,
    max_tokens=256,
    messages=[
        {"role": "user", "content": "Name three real-world applications of agentic AI."}
    ],
)

print("--- TEXT ---")
print(response.content[0].text)

print("\n--- METADATA ---")
print(f"Model        : {response.model}")
print(f"Stop reason  : {response.stop_reason}")
print(f"Input tokens : {response.usage.input_tokens}")
print(f"Output tokens: {response.usage.output_tokens}")

# Cost estimate (Haiku pricing, approximate)
cost = (response.usage.input_tokens * 0.80 + response.usage.output_tokens * 4.00) / 1_000_000
print(f"Approx cost  : ${cost:.6f} USD")

--- TEXT ---
# Three Real-World Applications of Agentic AI

1. **Customer Service Automation**
   Agentic AI systems handle complex support tickets by independently researching solutions, checking knowledge bases, contacting relevant departments, and resolving issues without constant human intervention. They can escalate when needed while managing routine cases end-to-end.

2. **Research and Data Analysis**
   AI agents autonomously gather information from multiple sources, synthesize findings, run analyses, and generate reports. This is used in competitive intelligence, scientific research, market analysis, and due diligence where agents work over extended periods to compile comprehensive insights.

3. **Software Development and DevOps**
   Agentic AI systems can autonomously write code, run tests, debug issues, deploy applications, and monitor systems. They handle tasks like creating pull requests, fixing failing tests, and managing infrastructure changes with minimal human oversight

### Multi-turn conversation
Add previous turns to `messages` to give the model context about the conversation.

In [7]:
conversation = [
    {"role": "user",      "content": "What is machine learning?"},
    {"role": "assistant", "content": "Machine learning is a way for computers to learn patterns from data without being explicitly programmed for each task."},
    {"role": "user",      "content": "How is that different from an AI agent?"},
]

response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    messages=conversation,
)

print(response.content[0].text)
print("\n[Notice: the model referenced 'machine learning' from earlier — that's memory via conversation history]")

# Key Differences

**Machine Learning** is a technical approach—a set of algorithms and techniques that learn patterns from data. It's a tool or method.

**AI Agent** is a broader concept—a system (which may or may not use machine learning) that:
- Perceives its environment
- Makes decisions
- Takes actions to achieve goals
- Often acts autonomously

## The Relationship

- An AI agent *can use* machine learning as one of its components
- But an agent doesn't *have* to use ML—it could use rule-based logic, search algorithms, or other approaches
- Conversely, machine learning systems don't necessarily act as autonomous agents

## Simple Example

- **ML only**: A spam filter that learns which emails are spam by analyzing past examples
- **AI Agent**: A robot vacuum that uses ML to recognize obstacles, but also autonomously decides where to go and when to clean

Think of it this way: machine learning is about *learning from data*, while an AI agent is about *taking intelligent action in th

---
# Part 2 — System Prompt Engineering

## The system prompt is the agent's identity card

The same model, same question — but a different system prompt produces dramatically different output. Every professional agent needs four things in its system prompt:

```
ROLE        — who the agent IS
GOAL        — what it optimises for
CONSTRAINTS — what it must NEVER do
FORMAT      — how it should structure its response
```

We'll run the same question through three system prompts and compare outputs.

In [8]:
QUESTION = "What causes inflation and how does it affect students?"

PROMPTS = {
    "1. Generic (no engineering)": (
        "You are a helpful assistant."
    ),

    "2. Role + Goal (better)": (
        "You are an expert economics tutor at a top university. "
        "Your goal is to explain complex economic concepts clearly "
        "to first-year students who have no prior economics background. "
        "Use analogies and everyday examples."
    ),

    "3. Role + Goal + Constraints + Format (best)": (
        "You are an expert economics tutor at a top university.\n\n"
        "GOAL: Help first-year students understand economics using "
        "simple language, relatable analogies, and concrete examples.\n\n"
        "CONSTRAINTS:\n"
        "- Never use jargon without immediately explaining it\n"
        "- Never give advice about personal finances or investments\n"
        "- Never claim certainty on contested economic topics\n\n"
        "OUTPUT FORMAT:\n"
        "1. One-sentence plain-English definition\n"
        "2. How it works (3 bullet points max)\n"
        "3. What it means for students specifically\n"
        "4. One analogy that makes it click"
    ),
}

results = {}
for name, system_prompt in PROMPTS.items():
    response = client.messages.create(
        model=MODEL,
        max_tokens=500,
        system=system_prompt,
        messages=[{"role": "user", "content": QUESTION}],
    )
    results[name] = response.content[0].text
    print(f"  Called: {name}")

print("\nAll responses collected. Run the next cell to compare.")

  Called: 1. Generic (no engineering)
  Called: 2. Role + Goal (better)
  Called: 3. Role + Goal + Constraints + Format (best)

All responses collected. Run the next cell to compare.


In [9]:
separator = "=" * 70

for name, answer in results.items():
    print(f"\n{separator}")
    print(f"  PROMPT: {name}")
    print(separator)
    print(answer)

print(f"\n{separator}")
print("  PROMPT LENGTH vs OUTPUT QUALITY")
print(separator)
for name, prompt in PROMPTS.items():
    words = len(prompt.split())
    print(f"  {name[:50]:<50}  {words:>3} words in prompt")

print("""
  TAKEAWAY:
  Spending ~50 extra words on a system prompt reliably produces
  more useful, safer, and more consistent agent outputs.
  Better prompts are free — they just require engineering.
""")


  PROMPT: 1. Generic (no engineering)
# Causes of Inflation

**Main drivers include:**
- **Increased money supply** – More money chasing the same goods
- **Supply chain disruptions** – Fewer goods available (raising prices)
- **Rising production costs** – Higher wages, raw materials, or energy
- **Demand surge** – People buying more than supply can meet
- **Inflation expectations** – Consumers/businesses anticipating price increases, causing them

# How Inflation Affects Students

**Negative impacts:**
- **Higher education costs** – Tuition, books, and housing become more expensive
- **Living expenses** – Food, transportation, and utilities cost more
- **Reduced purchasing power** – Money doesn't stretch as far
- **Student debt burden** – If loans have fixed interest rates, the real value of repayment increases
- **Job market pressure** – May need to work more hours while studying

**Possible benefits:**
- **Wage increases** – If student wages rise with inflation
- **Existing debt eas

### Exercise 2.1 — Write your own system prompt
Edit the cell below. Change the role, goal, and constraints. Run it and observe the difference.

In [10]:
# TODO: Edit this system prompt and observe how the output changes
MY_SYSTEM_PROMPT = """
You are _____ (fill in a role).

GOAL: _____ (fill in what to optimise for)

CONSTRAINTS:
- _____

OUTPUT FORMAT:
- _____
"""

MY_QUESTION = "Explain how the internet works to a 10-year-old."

response = client.messages.create(
    model=MODEL,
    max_tokens=400,
    system=MY_SYSTEM_PROMPT,
    messages=[{"role": "user", "content": MY_QUESTION}],
)
print(response.content[0].text)

# How the Internet Works (Explained Simply!)

Imagine the internet like a **giant network of connected computers** all talking to each other. Here's how it works:

## The Basic Idea

When you want to visit a website or send a message:

1. **Your device sends a request** (like asking "Can I see YouTube?")
2. **It travels through cables** (under streets, under the ocean, through the air)
3. **It reaches the website's computer** (called a "server")
4. **The server sends back what you asked for** (the video, the page, etc.)
5. **Your computer receives it** and shows it on your screen

## Think of It Like Mail

The internet works a lot like the postal system:
- **Your computer** = Your house
- **The website** = Your friend's house
- **Your message** = A letter
- **The cables** = Mail trucks and roads
- **Your Internet Provider** = The post office that sends your letter

## What Makes It All Work?

- **Cables & Signals**: Information travels through cables (like fiber optic) and wireless sig

---
# Part 3 — The PPARM Framework

## What separates a chatbot from an agent?

A **chatbot** answers one question.  
An **agent** perceives, plans, acts, reflects, and remembers — across multiple steps.

```
P  — PERCEPTION   How the agent receives and structures input
P  — PLANNING     How it breaks the task into steps (CoT lives here)
A  — ACTION       How it executes (LLM call, tool call, code run)
R  — REFLECTION   How it judges its own output
M  — MEMORY       How it stores and retrieves past context
```

We'll run a live API call for each phase so you can see what it does.

> **Why RAG alone isn't enough**  
> RAG retrieves facts. An agent *does things* — it plans, decides, loops, and calls tools. RAG is one piece of the Memory phase, not a replacement for reasoning.

In [11]:
import datetime

QUESTION = "Explain Newton's three laws of motion with a real-world example for each."

# ── P: PERCEPTION ─────────────────────────────────────────────────────────────
# The agent receives raw input and turns it into structured data.
# In a real system this might include image parsing, audio transcription, etc.

def perceive(raw_input: str) -> dict:
    perception = {
        "raw_input"  : raw_input,
        "type"       : "assignment_question",
        "word_count" : len(raw_input.split()),
        "timestamp"  : datetime.datetime.now().isoformat(),
    }
    print("\n" + "=" * 60)
    print("  P — PERCEPTION")
    print("=" * 60)
    for k, v in perception.items():
        print(f"  {k}: {v}")
    return perception

perception = perceive(QUESTION)


  P — PERCEPTION
  raw_input: Explain Newton's three laws of motion with a real-world example for each.
  type: assignment_question
  word_count: 12
  timestamp: 2026-05-04T06:34:01.506130


In [12]:
# ── P: PLANNING ───────────────────────────────────────────────────────────────
# Before acting, a smart agent breaks the task into steps.
# Planning first and acting once is cheaper than acting blindly and retrying.

def plan(perception: dict) -> str:
    print("\n" + "=" * 60)
    print("  P — PLANNING (Chain-of-Thought)")
    print("=" * 60)

    response = client.messages.create(
        model=MODEL,
        max_tokens=400,
        system=(
            "You are a planning assistant. "
            "Given a task, produce a numbered list of 3-5 steps to answer it well. "
            "Be concise — one line per step."
        ),
        messages=[{"role": "user", "content": f"Task: {perception['raw_input']}"}],
    )
    plan_text = response.content[0].text
    print(plan_text)
    return plan_text

plan_text = plan(perception)


  P — PLANNING (Chain-of-Thought)
# Newton's Three Laws of Motion

1. **Define Newton's First Law (Inertia)** — An object at rest stays at rest, and an object in motion stays in motion unless acted upon by a force.
   - *Example: A car passenger lurches forward when brakes are applied because their body continues moving forward while the car stops.*

2. **Define Newton's Second Law (F=ma)** — The acceleration of an object is directly proportional to the net force applied and inversely proportional to its mass.
   - *Example: Pushing an empty shopping cart requires less force than pushing a full one to achieve the same acceleration.*

3. **Define Newton's Third Law (Action-Reaction)** — For every action, there is an equal and opposite reaction.
   - *Example: A swimmer pushes water backward with their hands, and the water pushes the swimmer forward with equal force.*


In [13]:
# ── A: ACTION ─────────────────────────────────────────────────────────────────
# The agent executes its plan.
# Day 1: one LLM call.
# Day 2: LLM call + real tool calls (web search, calculator, etc.).

def act(perception: dict, plan_text: str) -> str:
    print("\n" + "=" * 60)
    print("  A — ACTION")
    print("=" * 60)
    print("  Calling Claude with the plan as context...")

    response = client.messages.create(
        model=MODEL,
        max_tokens=800,
        system=(
            "You are a university assignment assistant. "
            "Follow the provided plan exactly. "
            "Write clearly for a first-year university student."
        ),
        messages=[{
            "role": "user",
            "content": (
                f"Plan to follow:\n{plan_text}\n\n"
                f"Question to answer:\n{perception['raw_input']}"
            ),
        }],
    )
    answer = response.content[0].text
    print(f"  Tokens: {response.usage.input_tokens} in / {response.usage.output_tokens} out")
    print("\n" + answer)
    return answer

answer = act(perception, plan_text)


  A — ACTION
  Calling Claude with the plan as context...
  Tokens: 255 in / 540 out

# Newton's Three Laws of Motion

## 1. Newton's First Law: The Law of Inertia

**Definition:** An object at rest stays at rest, and an object in motion stays in motion unless acted upon by an external force.

In simpler terms, objects don't naturally start or stop moving on their own—they need a force to make them change what they're doing.

**Real-World Example: Car Braking**

Imagine you're sitting in a car traveling at 60 km/h. The car suddenly brakes hard. Your body lurches forward violently, even though the car has stopped. Why? Your body wants to keep moving forward (inertia) because nothing is directly pushing you backward—only the car stopped. The seatbelt provides the force needed to decelerate your body along with the car. Without it, you'd continue moving forward into the windshield!

---

## 2. Newton's Second Law: Force Equals Mass Times Acceleration (F = ma)

**Definition:** The acceler

In [14]:
# ── R: REFLECTION ─────────────────────────────────────────────────────────────
# The agent evaluates its own output.
# A low score could trigger a retry or escalate to a more powerful model.

def reflect(answer: str, original_question: str) -> dict:
    print("\n" + "=" * 60)
    print("  R — REFLECTION (self-critique)")
    print("=" * 60)

    response = client.messages.create(
        model=MODEL,
        max_tokens=200,
        system=(
            "You are a strict academic evaluator. "
            "Score the answer 1-10 and give ONE specific improvement suggestion. "
            "Format: Score: X/10 | Suggestion: <one sentence>"
        ),
        messages=[{"role": "user", "content": f"Question: {original_question}\n\nAnswer:\n{answer}"}],
    )
    feedback = response.content[0].text
    print(f"  {feedback}")
    return {"feedback": feedback, "answer": answer}

result = reflect(answer, QUESTION)


  R — REFLECTION (self-critique)
  Score: 9/10 | Suggestion: In the Second Law example, explicitly calculate or compare the forces numerically (e.g., "pushing with 50 N on an empty 10 kg cart gives 5 m/s² acceleration, but the same force on a 50 kg full cart gives only 1 m/s²") to make the F=ma relationship more concrete and memorable.


In [16]:
# ── M: MEMORY ─────────────────────────────────────────────────────────────────
# Day 1: simple Python list (in-process, lost when session ends).
# Day 3: state dict flowing through LangGraph.
# Day 4: ChromaDB vector store — persists across sessions.

_session_memory = []

def remember(perception: dict, result: dict) -> None:
    _session_memory.append({
        "question" : perception["raw_input"],
        "answer"   : result["answer"][:120] + "...",
        "saved_at" : datetime.datetime.now().isoformat(),
    })
    print("\n" + "=" * 60)
    print("  M — MEMORY")
    print("=" * 60)
    print(f"  Stored entry #{len(_session_memory)} in session memory.")
    print(f"  Total items in memory: {len(_session_memory)}")
    print(f"  Preview: {_session_memory[-1]['answer'][:80]}...")

remember(perception, result)

print("""
KEY INSIGHT
-----------
Without planning (P), the action (A) is often shallow.
Without reflection (R), bad answers are never caught.
Without memory (M), the agent forgets everything on restart.

Day 2 adds real tools to the Action phase.
Day 3 splits each phase across specialised agents.
Day 4 upgrades Memory from a list to a vector database.
""")


  M — MEMORY
  Stored entry #1 in session memory.
  Total items in memory: 1
  Preview: # Newton's Three Laws of Motion

## 1. Newton's First Law: The Law of Inertia

*...

KEY INSIGHT
-----------
Without planning (P), the action (A) is often shallow.
Without reflection (R), bad answers are never caught.
Without memory (M), the agent forgets everything on restart.

Day 2 adds real tools to the Action phase.
Day 3 splits each phase across specialised agents.
Day 4 upgrades Memory from a list to a vector database.



---
# Part 4 — Chain-of-Thought vs Direct Answering

## Why does reasoning before answering help?

When a model is forced to reason step-by-step *before* concluding, it makes fewer errors — especially on tasks that require logic, trade-offs, or multiple constraints.

```
Direct CoT:     Question → Answer  (fast, often shallow)

Basic CoT:      Question → Think step by step → Answer  (better)

Structured CoT: Question → ANALYSIS → REASONING → RECOMMENDATION → CONFIDENCE
                (predictable sections → easy to parse in code)
```

### Chain-of-Thought vs Tree-of-Thought

```
CoT  = one chain:   Thought1 → Thought2 → Thought3 → Answer

ToT  = branching:   Branch A → Dead end (backtrack)
                    Branch B → Dead end (backtrack)
                    Branch C → Answer ✓
```

**Use ToT when:** the problem has multiple plausible strategies, mistakes early invalidate the chain, or you need verifiable correctness (math proofs, logic puzzles).  
For this bootcamp, **Structured CoT** is enough — it's cheaper, reliable, and parseable.

In [17]:
COT_QUESTION = (
    "A student has 3 assignments due. "
    "Essay (8hrs, worth 40%), Lab report (3hrs, worth 20%), Quiz (1hr, worth 10%). "
    "They have 6 hours left. What should they do and why?"
)

DIRECT_SYSTEM = (
    "You are a university advisor. Answer student questions clearly and concisely."
)

BASIC_COT_SYSTEM = (
    "You are a university advisor. "
    "Before answering, think step by step. "
    "Show your reasoning, then give a final recommendation."
)

STRUCTURED_COT_SYSTEM = (
    "You are a university advisor. Always respond in this exact format:\n\n"
    "ANALYSIS:\n"
    "<break down the constraints and trade-offs>\n\n"
    "REASONING:\n"
    "<apply the constraints to rank options>\n\n"
    "RECOMMENDATION:\n"
    "<specific, actionable advice — what to do and in what order>\n\n"
    "CONFIDENCE: <High/Medium/Low — and why>"
)

styles = [
    ("1. Direct (no CoT)",     DIRECT_SYSTEM),
    ("2. Basic CoT",           BASIC_COT_SYSTEM),
    ("3. Structured CoT",      STRUCTURED_COT_SYSTEM),
]

cot_results = {}
for name, system in styles:
    response = client.messages.create(
        model=MODEL, max_tokens=600,
        system=system,
        messages=[{"role": "user", "content": COT_QUESTION}],
    )
    cot_results[name] = (response.content[0].text, response.usage.output_tokens)
    print(f"  Called: {name}")

print("\nRun the next cell to see the comparison.")

  Called: 1. Direct (no CoT)
  Called: 2. Basic CoT
  Called: 3. Structured CoT

Run the next cell to see the comparison.


In [18]:
print(f"Question: {COT_QUESTION}\n")

for name, (answer, tokens) in cot_results.items():
    print("=" * 70)
    print(f"  {name}")
    print("=" * 70)
    print(answer)
    print(f"  [Output tokens: {tokens}]")

print("\n" + "=" * 70)
print("  TOKEN USAGE COMPARISON")
print("=" * 70)
for name, (_, tokens) in cot_results.items():
    bar = "█" * (tokens // 10)
    print(f"  {name[:30]:<30} {tokens:>4} tokens  {bar}")

print("""
NOTICE:
- Direct:         Fast, but may miss key trade-offs.
- Basic CoT:      Reasons more carefully, but format varies.
- Structured CoT: Predictable sections you can parse in code.
                  The pipeline can extract RECOMMENDATION
                  without an extra LLM call.

For agents, predictable output format > brevity.
""")

Question: A student has 3 assignments due. Essay (8hrs, worth 40%), Lab report (3hrs, worth 20%), Quiz (1hr, worth 10%). They have 6 hours left. What should they do and why?

  1. Direct (no CoT)
# Time Management Recommendation

**Prioritize: Essay → Lab Report → Quiz**

## Why This Order?

1. **Essay (8hrs, 40% weight)** - Start immediately
   - Highest value-to-time ratio (40% ÷ 8 = 5% per hour)
   - Most impactful on your grade
   - Spend your full 6 hours here; aim to complete as much as possible

2. **Lab Report (3hrs, 20% weight)** - Do next if time allows
   - Second-best return (20% ÷ 3 = 6.7% per hour)
   - Shorter, so more achievable if you finish the essay

3. **Quiz (1hr, 10% weight)** - Lower priority
   - Lowest value-to-time ratio (10% ÷ 1 = 10% per hour)
   - Quick review only if time remains

## Action Plan
- Use all 6 hours on the essay to maximize completion
- A partial essay is better than skipping it entirely
- Only pivot to the lab report if you finish the essay 

### Parsing structured CoT output in code
Because we defined labelled sections, we can extract just the RECOMMENDATION without prompting again.

In [19]:
structured_answer = cot_results["3. Structured CoT"][0]

def extract_section(text: str, section: str) -> str:
    """Extract a named section from structured CoT output."""
    lines = text.split("\n")
    capturing = False
    result = []
    for line in lines:
        if line.strip().startswith(section + ":"):
            capturing = True
            continue
        if capturing:
            # Stop at the next all-caps section header
            if line.strip() and line.strip()[0].isupper() and line.strip().endswith(":"):
                break
            result.append(line)
    return "\n".join(result).strip()

recommendation = extract_section(structured_answer, "RECOMMENDATION")
confidence     = extract_section(structured_answer, "CONFIDENCE")

print("Extracted RECOMMENDATION:")
print(recommendation)
print(f"\nExtracted CONFIDENCE: {confidence}")
print("\n[This is how agents parse their own outputs — no extra LLM call needed]")

Extracted RECOMMENDATION:
1. **Spend 1 hour on the quiz** — complete it fully
2. **Spend 3 hours on the lab report** — aim for finished (not perfect)
3. **Spend 2 hours on the essay** — write a strong intro, thesis, one fully developed argument, brief conclusion
4. Submit all three rather than perfecting one or two

Expected grade impact: ~38-42% vs. ~20-25% if you'd attempted everything equally.

CONFIDENCE: **High** — This balances mathematical optimization (ROI) with realistic academic grading practices (partial credit) and human factors (momentum from quick wins).

Extracted CONFIDENCE: 

[This is how agents parse their own outputs — no extra LLM call needed]


---
# Part 5 — Lab: Build Assignment Agent v1

## Your deliverable

A complete agent that:
1. Accepts any assignment question
2. Uses **CoT** to plan step by step
3. Executes all **5 PPARM phases**
4. Ends with a **self-critique score**
5. **Remembers** previous questions this session

The code below is your scaffold. Read it, understand each section, then run it.

**After the bootcamp:**
- Day 2: add tool use (web search, calculator)
- Day 3: split into 4 specialised LangGraph agents
- Day 4: upgrade memory from a list → ChromaDB
- Day 5: wrap in FastAPI + WhatsApp interface + Docker

In [20]:
import json

# ── Agent identity ─────────────────────────────────────────────────────────────
AGENT_NAME    = "AssignmentBot v1"
AGENT_VERSION = "1.0.0"

SYSTEM_PROMPT = """\
You are AssignmentBot — an expert academic assistant for university students.

ROLE: Help students understand and answer their assignments at a deep level.

GOAL: Produce clear, accurate, well-structured answers that help the student
genuinely understand the topic, not just copy an answer.

CONSTRAINTS:
- Never write an assignment *for* the student verbatim — explain and guide.
- Never claim certainty on contested academic topics; signal uncertainty.
- Never give medical, legal, or financial advice even if asked as homework.
- Always cite reasoning, not just conclusions.

OUTPUT FORMAT:
## Understanding the Question
<what the question is really asking>

## Step-by-Step Answer
<numbered steps, each with explanation>

## Key Takeaway
<one sentence the student should remember>
"""


# ── Memory (Day 1: in-process list; Day 4: upgrades to ChromaDB) ───────────────
class SessionMemory:
    def __init__(self):
        self._store = []

    def save(self, question: str, answer: str, score: int) -> None:
        self._store.append({
            "id"       : len(self._store) + 1,
            "question" : question,
            "answer"   : answer[:200] + "..." if len(answer) > 200 else answer,
            "score"    : score,
            "timestamp": datetime.datetime.now().isoformat(),
        })

    def recall_recent(self, n: int = 3) -> list:
        return self._store[-n:]

    def __len__(self):
        return len(self._store)


memory = SessionMemory()


# ── PPARM phases ───────────────────────────────────────────────────────────────

def perceive_v1(raw_input: str) -> dict:
    return {
        "raw_input"  : raw_input.strip(),
        "word_count" : len(raw_input.split()),
        "timestamp"  : datetime.datetime.now().isoformat(),
    }


def plan_v1(perception: dict) -> str:
    response = client.messages.create(
        model=MODEL,
        max_tokens=300,
        system=(
            "You are a planning assistant. Given an assignment question, "
            "list 3-5 concrete steps to answer it thoroughly. One line per step."
        ),
        messages=[{"role": "user", "content": f"Assignment question: {perception['raw_input']}"}],
    )
    return response.content[0].text


def act_v1(perception: dict, plan_text: str, recent_context: list) -> str:
    context_block = ""
    if recent_context:
        context_block = "\n\nPrevious questions this session (for context only):\n"
        for item in recent_context:
            context_block += f"- {item['question']}\n"

    response = client.messages.create(
        model=MODEL,
        max_tokens=1000,
        system=SYSTEM_PROMPT,
        messages=[{
            "role": "user",
            "content": (
                f"Plan:\n{plan_text}"
                f"{context_block}\n\n"
                f"Question: {perception['raw_input']}"
            ),
        }],
    )
    return response.content[0].text


def reflect_v1(answer: str, question: str):
    response = client.messages.create(
        model=MODEL,
        max_tokens=150,
        system=(
            "You are a strict academic evaluator. "
            "Reply in this exact format only:\n"
            "Score: X/10\n"
            "Issue: <one specific problem or 'None'>\n"
            "Fix: <one-sentence improvement or 'N/A'>"
        ),
        messages=[{"role": "user", "content": f"Question: {question}\n\nAnswer:\n{answer}"}],
    )
    feedback = response.content[0].text
    try:
        score_line = [l for l in feedback.splitlines() if l.startswith("Score:")][0]
        score = int(score_line.split("/")[0].replace("Score:", "").strip())
    except (IndexError, ValueError):
        score = 7
    return feedback, score


# ── Full agent pipeline ────────────────────────────────────────────────────────

def run_agent(question: str) -> None:
    print(f"\n{'=' * 60}")
    print(f"  {AGENT_NAME}  (v{AGENT_VERSION})")
    print(f"{'=' * 60}")
    print(f"  Question: {question}")
    print()

    print("  [1/5] Perceiving...")
    p = perceive_v1(question)

    print("  [2/5] Planning (CoT)...")
    plan_text = plan_v1(p)
    print(f"        Steps: {plan_text.splitlines()[0]}...")

    print("  [3/5] Acting...")
    context = memory.recall_recent(3)
    answer  = act_v1(p, plan_text, context)

    print("  [4/5] Reflecting...")
    feedback, score = reflect_v1(answer, question)

    print("  [5/5] Storing in memory...")
    memory.save(question, answer, score)

    print(f"\n{'─' * 60}")
    print(answer)
    print(f"{'─' * 60}")
    print(f"  Self-critique: {feedback}")
    print(f"  Memory: {len(memory)} item(s) stored this session.")
    print(f"{'─' * 60}")


print(f"{AGENT_NAME} ready. Run the next cells to use it.")

AssignmentBot v1 ready. Run the next cells to use it.


In [21]:
# Ask the agent a question
run_agent("Explain the difference between supervised and unsupervised machine learning with examples.")


  AssignmentBot v1  (v1.0.0)
  Question: Explain the difference between supervised and unsupervised machine learning with examples.

  [1/5] Perceiving...
  [2/5] Planning (CoT)...
        Steps: # Steps to Answer This Assignment...
  [3/5] Acting...
  [4/5] Reflecting...
  [5/5] Storing in memory...

────────────────────────────────────────────────────────────
# Understanding the Question

The question asks you to:
1. Distinguish between two fundamental ML paradigms
2. Explain *how* and *why* they differ (not just name differences)
3. Ground your explanation with concrete, real-world examples

This tests whether you understand the underlying logic of each approach, not just definitions.

---

# Step-by-Step Answer

## 1. Define Supervised Learning

**Supervised learning** trains models using **labeled data** — datasets where each input has a corresponding correct output.

- The model learns a mapping: Input → Output
- "Supervised" because the correct answers guide (supervise) the lea

In [22]:
# Ask a second question — the agent will use memory from the first question as context
run_agent("How does overfitting relate to the concepts you just explained?")


  AssignmentBot v1  (v1.0.0)
  Question: How does overfitting relate to the concepts you just explained?

  [1/5] Perceiving...
  [2/5] Planning (CoT)...
        Steps: # Steps to Answer: Overfitting and Related Concepts...
  [3/5] Acting...
  [4/5] Reflecting...
  [5/5] Storing in memory...

────────────────────────────────────────────────────────────
## Understanding the Question

This question asks you to connect **overfitting** to machine learning fundamentals—specifically how it emerges from the supervised learning process and why it's a critical problem. The phrasing "relate to the concepts you just explained" suggests linking overfitting to supervised vs. unsupervised learning, model training, and generalization.

---

## Step-by-Step Answer

### 1. **Overfitting in the Context of Supervised Learning**

Since you just covered supervised learning, here's the connection:
- **Supervised learning** trains models on labeled data (input-output pairs) to learn a mapping function
- **O

In [24]:
# Inspect what's in session memory
print("Session Memory Contents:")
print("=" * 60)
for item in memory.recall_recent(10):
    print(f"[{item['id']}] Score {item['score']}/10 | {item['question'][:60]}")
    print(f"     Saved at: {item['timestamp']}")
    print()

Session Memory Contents:
[1] Score 9/10 | Explain the difference between supervised and unsupervised m
     Saved at: 2026-05-04T06:36:41.258848

[2] Score 9/10 | How does overfitting relate to the concepts you just explain
     Saved at: 2026-05-04T06:36:58.437643



### Your turn — ask your own question

In [25]:
# TODO: Replace this with any assignment question you want to test
YOUR_QUESTION = "What is the difference between correlation and causation? Give a real example."

run_agent(YOUR_QUESTION)


  AssignmentBot v1  (v1.0.0)
  Question: What is the difference between correlation and causation? Give a real example.

  [1/5] Perceiving...
  [2/5] Planning (CoT)...
        Steps: # Steps to Answer This Assignment...
  [3/5] Acting...
  [4/5] Reflecting...
  [5/5] Storing in memory...

────────────────────────────────────────────────────────────
# Understanding the Question

You're being asked to distinguish between two fundamental statistical/scientific concepts: **correlation** (statistical association) and **causation** (direct causal mechanism). The "real example" part requires you to show *why* this distinction matters in practice.

---

# Step-by-Step Answer

## 1. **Define Correlation**

Correlation measures how two variables *move together*. When one increases, does the other tend to increase (positive correlation), decrease (negative correlation), or show no pattern (no correlation)?

- Measured on a scale from **-1 to +1**
- **+1** = perfect positive relationship
- **-1*

---
## Challenge Exercises (if you finish early)

**Level 1 — Tweak the agent identity**  
Edit `SYSTEM_PROMPT` above. Change the role to a "debate coach" or "Socratic tutor" and re-run. What changes?

**Level 2 — Add a retry loop**  
If the reflection score is below 7, automatically re-run the Action phase. Add this logic to `run_agent()`.

```python
# Hint: add this after reflect_v1()
if score < 7:
    print("  Score too low — retrying action...")
    answer = act_v1(p, plan_text, context)
    feedback, score = reflect_v1(answer, question)
```

**Level 3 — Subject-aware routing**  
Detect whether the question is maths, science, or humanities. Use a different system prompt for each subject.

---
# Checkpoint — Demo to the Class

Run the cell below. This is your 2-minute demo for the class.

Show:
1. All 5 PPARM phases printing in sequence
2. The structured answer output
3. The self-critique score
4. Memory growing across two questions

In [26]:
# Checkpoint demo — two questions in sequence to show memory
demo_memory = SessionMemory()

def run_agent_demo(question: str) -> None:
    p           = perceive_v1(question)
    plan_text   = plan_v1(p)
    context     = demo_memory.recall_recent(3)
    answer      = act_v1(p, plan_text, context)
    feedback, s = reflect_v1(answer, question)
    demo_memory.save(question, answer, s)

    print(f"\n{'=' * 60}")
    print(f"  Q: {question}")
    print(f"{'=' * 60}")
    print(answer[:800] + ("..." if len(answer) > 800 else ""))
    print(f"\n  Critique: {feedback}")
    print(f"  Memory: {len(demo_memory)} items stored")

print("DEMO — AssignmentBot v1")
print("=" * 60)

run_agent_demo("What is the water cycle and why does it matter for climate?")
run_agent_demo("How does the water cycle connect to the greenhouse effect?")

DEMO — AssignmentBot v1

  Q: What is the water cycle and why does it matter for climate?
# Understanding the Question

You're being asked to explain **what** the water cycle is (the mechanism) and **why** it's important to climate science (the significance). This requires both mechanism and consequence—don't just describe the cycle; explain its climate relevance.

---

# Step-by-Step Answer

## 1. **Define the Water Cycle with Clear Processes**

Start by naming and explaining the four main stages:

- **Evaporation**: Water from oceans, lakes, and soil surfaces absorbs solar energy and transforms into water vapor (gas), rising into the atmosphere.
- **Condensation**: As water vapor rises and cools at higher altitudes, it changes back into liquid droplets, forming clouds.
- **Precipitation**: Water falls as rain, snow, sleet, or hail when clouds become heavy enough.
- **Collecti...

  Critique: Score: 9/10

Issue: The response is a study guide/teaching framework rather than a direct ans

---
# Day 1 Summary

## What you built
- Made direct API calls to Claude (the foundation of everything)
- Engineered system prompts with Role, Goal, Constraints, and Format
- Implemented the full **PPARM loop** — 5 phases, 4 API calls, one agent
- Compared **Direct vs CoT vs Structured CoT** output quality
- Built and ran **AssignmentBot v1**

## Key vocabulary

| Term | Definition |
|------|------------|
| LLM | Large Language Model — predicts tokens given context |
| System prompt | Developer instructions that define agent identity |
| PPARM | Perception, Planning, Action, Reflection, Memory |
| Chain-of-Thought | Prompt technique: reason before concluding |
| Structured CoT | CoT with labelled sections — parseable in code |
| Session memory | In-process list — lost when notebook restarts |

## What's next — Day 2

> We give the agent **hands**.  
> Instead of only calling the LLM, it will call real tools: web search, a calculator, a file reader.  
> You will implement the **ReAct loop** (Reason → Act → Observe → Repeat) and watch the agent iterate until it finds the answer.

---
*Agentic AI Bootcamp — Day 1 complete*